# 01 — Build Master Dataset  
## Forecasting and Identifying Global Export Opportunities for Algerian Exporters  
**ENSIA — Machine Learning Project — Spring 2025/2026**

---

## Notebook Purpose

This notebook builds the first version of the integrated master dataset used in the project.  
The main objective is to transform raw international trade data into a clean machine-learning-ready dataset that can later be used for:

- export opportunity classification,
- country/product clustering,
- forecasting trade trends,
- exploratory data analysis and dashboard visualizations.

This notebook mainly covers **Step 01: Data Collection** and **Step 02: Data Preparation & Integration** from the project methodology.

---

## What this notebook does

The pipeline starts from raw BACI bilateral trade files and produces a final dataset called:

`data/master_df.parquet`

The main steps are:

1. Load BACI bilateral trade data from 2012 to 2023.
2. Load country and product code reference tables.
3. Attach ISO3 country codes to partner countries.
4. Build component dataframes:
   - Algeria's exports by partner and product,
   - partner imports by product,
   - world imports by product,
   - Algeria's imports from each partner.
5. Build a full opportunity grid containing all:
   - years,
   - partner countries,
   - HS6 products.

This is important because the model should not only learn from markets where Algeria already exports, but also from markets where Algeria currently exports zero.

6. Compute trade and economic features such as:
   - export growth,
   - world demand growth,
   - global demand index,
   - market penetration,
   - trade balance,
   - revealed comparative advantage (RCA),
   - number of export destinations.

7. Compute lag features using previous years.
8. Define the target variable `label_opportunity`.
9. Merge external indicators when available:
   - CEPII GeoDist variables,
   - World Bank macroeconomic indicators,
   - UNCTAD diversification indicators.
10. Add product descriptions.
11. Create a temporal train / validation / test split.
12. Define simple and advanced feature sets.
13. Run sanity checks.
14. Save the final master dataset.

---

## Dataset Grain

Each row represents one:

**(year, partner country, HS6 product)**

In other words, each observation corresponds to a specific product exported or potentially exported by Algeria to a specific country in a specific year.

The dataset also includes zero-export pairs.  
These rows are important because they represent possible missed or future export opportunities.

---

## Data Sources

| Source | Role in this notebook |
|---|---|
| BACI HS12 | Main bilateral trade data source: exports/imports by country, product, and year |
| BACI country/product codes | Mapping numeric country/product codes to readable labels |
| CEPII GeoDist | Distance, common language, border, and historical relationship variables when available |
| World Bank API | GDP, population, trade openness, GDP per capita, and market size |
| UNCTAD | Algeria-level export concentration and diversification indicators |

---

## Relation to the Project Requirements

This notebook supports the project requirement of building an integrated multi-source international trade dataset.  
It also prepares the main engineered variables required for later machine learning tasks, including classification, clustering, forecasting, and dashboard analysis.

The final output of this notebook will be used in the next notebooks for exploratory data analysis, modeling, ranking export opportunities, and visualization.

---

## 0. Imports and Configuration

In this first step, we import the Python libraries required for data processing and define the main constants used throughout the notebook.

The configuration section is important because it centralizes all paths, years, country codes, and split boundaries in one place. This makes the notebook easier to modify later if the data folder, BACI version, or study period changes.

Main parameters used in this notebook:

- `BACI_DIR`: folder containing the raw BACI yearly trade files.
- `BACI_VERSION`: version of the BACI files used in this project.
- `HS_VERSION`: product classification version used by BACI.
- `ALG_CODE = 12`: Algeria's numeric country code in BACI.
- `ALG_ISO3 = "DZA"`: Algeria's ISO3 code, used when merging with external datasets.
- `YEARS = 2012–2023`: study period used to build the panel dataset.
- `MIN_WORLD_DEMAND_USD = 1_000`: minimum global demand threshold.  
  Since BACI trade values are expressed in **thousand USD**, this corresponds to approximately **1 million USD**.
- `TRAIN_END` and `VAL_END`: chronological split boundaries used to create train, validation, and test sets without future data leakage.

This configuration helps ensure that the whole pipeline is reproducible and consistent.

In [ ]:
import numpy as np
import pandas as pd
import requests
from pathlib import Path
import sys

try:
    sys.stdout.reconfigure(encoding="utf-8")
except AttributeError:
    pass

# ── Paths ────────────────────────────────────────────────────────────────────
# The notebook is inside the "notebooks" folder, so we use ../ to go back to the project folder.
BACI_DIR      = Path("../data/baci")
BACI_VERSION  = "202601"
HS_VERSION    = "HS12"
GEODIST_PATH  = Path("../data/geodist/dist_cepii.xls")
UNCTAD_PATH   = Path("../data/unctad/unctad_diversification.csv")
OUTPUT_PATH   = Path("../data/master_df.parquet")

# ── Algeria identifiers ───────────────────────────────────────────────────────
ALG_CODE  = 12
ALG_ISO3  = "DZA"
YEARS     = list(range(2012, 2024))

# ── Grid filter ───────────────────────────────────────────────────────────────
# Keep only products whose peak annual world import value exceeds $1M
# BACI values are in thousand USD, so threshold = 1,000
MIN_WORLD_DEMAND_USD = 1_000

# ── Temporal split boundaries ─────────────────────────────────────────────────
TRAIN_END = 2019   # train:  2012–2019
VAL_END   = 2021   # val:    2020–2021  |  test: 2022–2023

print("Libraries loaded.")
print(f"Current working directory: {Path.cwd()}")
print(f"BACI directory exists: {BACI_DIR.exists()}")
print(f"Years covered: {YEARS[0]}–{YEARS[-1]}")
print(f"Output path:   {OUTPUT_PATH}")

Libraries loaded.
Current working directory: c:\Users\PC\Documents\machine-learning-project\machine-learning-project\notebooks
BACI directory exists: True
Years covered: 2012–2023
Output path:   ..\data\master_df.parquet


The configuration was loaded successfully.  
The study period covers 2012 to 2023, and the final integrated dataset will be saved as `data/master_df.parquet`.

I also define the temporal split boundaries here so that later modeling uses past years for training, intermediate years for validation, and the most recent years for testing. This avoids data leakage from the future.

---
## Step 1 — Load BACI Trade Data

BACI (Base pour l'Analyse du Commerce International) is produced by CEPII and provides harmonized bilateral trade flows at the HS6 product level.
Each row records: `i` (exporter), `j` (importer), `k` (HS6 product code), `t` (year), `v` (value in thousand USD), `q` (quantity in metric tonnes).

We load one CSV file per year and concatenate them into a single dataframe.
We also load two reference tables:
- `country_codes`: maps BACI numeric codes → ISO3 and country name
- `product_codes`: maps HS6 codes → product descriptions

In [ ]:
print("── Step 1: Loading BACI ──")

frames = []

for year in YEARS:
    path = BACI_DIR / f"BACI_{HS_VERSION}_Y{year}_V{BACI_VERSION}.csv"

    if path.exists():
        df = pd.read_csv(path, dtype={"k": str})
        frames.append(df)
        print(f"  Loaded {year}: {len(df):,} rows")
    else:
        print(f"  WARNING: {path} not found — skipping")

if len(frames) == 0:
    raise FileNotFoundError(
        "No BACI yearly files were found. Check BACI_DIR, BACI_VERSION, HS_VERSION, and file names."
    )

baci = pd.concat(frames, ignore_index=True)

print(f"\n  Total BACI rows: {len(baci):,}")

# Reference tables
country_codes = pd.read_csv(
    BACI_DIR / f"country_codes_V{BACI_VERSION}.csv",
    dtype={"country_code": int}
)

product_codes = pd.read_csv(
    BACI_DIR / f"product_codes_{HS_VERSION}_V{BACI_VERSION}.csv",
    dtype={"code": str}
)

# Build crosswalk: numeric code → ISO3 + country name
crosswalk = country_codes[["country_code", "country_iso3", "country_name"]].copy()
crosswalk.columns = ["country_code", "iso3", "country_name"]

print("\nReference tables loaded.")
print(f"  Country codes: {len(country_codes):,} rows")
print(f"  Product codes: {len(product_codes):,} rows")
print(f"  Crosswalk rows: {len(crosswalk):,}")

── Step 1: Loading BACI ──
  Loaded 2012: 9,012,155 rows
  Loaded 2013: 9,787,220 rows
  Loaded 2014: 10,205,742 rows
  Loaded 2015: 10,782,381 rows
  Loaded 2016: 10,868,820 rows
  Loaded 2017: 11,213,140 rows
  Loaded 2018: 11,370,143 rows
  Loaded 2019: 11,506,556 rows
  Loaded 2020: 11,150,663 rows
  Loaded 2021: 11,673,491 rows
  Loaded 2022: 11,677,171 rows
  Loaded 2023: 11,755,559 rows

  Total BACI rows: 131,003,041

Reference tables loaded.
  Country codes: 238 rows
  Product codes: 5,202 rows
  Crosswalk rows: 238


The BACI yearly trade files were loaded successfully for the full study period 2012–2023.  
The combined dataset contains 131,003,041 bilateral trade records, showing that the project is based on a large-scale real-world trade dataset rather than a small sample dataset.

The country and product reference tables were also loaded. These tables will be used later to attach ISO3 country codes, country names, and HS6 product descriptions to the master dataset.

## Step 2 — Attach ISO3 Country Codes

BACI uses numeric country codes (`i` for the exporter and `j` for the importer).  
Most external datasets, such as World Bank and GeoDist, use ISO3 country codes.

In this step, we create the `j → ISO3` mapping once and carry it through the rest of the pipeline.  
This avoids duplicate column errors later and ensures that all datasets use the same country identifier.

In [ ]:
print("── Step 2: Attaching ISO3 to all partner codes ──")

# Single crosswalk used for all downstream merges
j_to_iso3 = crosswalk[['country_code', 'iso3', 'country_name']].rename(
    columns={'country_code': 'j'}
)

print(f"  Crosswalk covers {len(j_to_iso3):,} country codes")
print(f"  Sample mapping:\n{j_to_iso3.head(3).to_string(index=False)}")

── Step 2: Attaching ISO3 to all partner codes ──
  Crosswalk covers 238 country codes
  Sample mapping:
 j iso3 country_name
 4  AFG  Afghanistan
 8  ALB      Albania
12  DZA      Algeria


The ISO3 mapping was created successfully.  
The crosswalk contains 238 country codes and confirms that Algeria's BACI code `12` correctly maps to `DZA`.

This mapping will be reused in later steps when merging the master dataset with external sources such as World Bank and GeoDist.

---
## Step 3 — Build Component Dataframes

We extract four aggregated tables from the raw BACI data. Each one captures a different dimension of trade:

| Table | Grain | Content |
|-------|-------|---------|
| `df_algeria` | (t, j, k) | Algeria's actual bilateral exports — only rows where flow > 0 |
| `df_partner_imports` | (t, j, k) | Each partner's total imports per product from all world sources |
| `df_world_imports` | (t, k) | Global demand per product per year |
| `df_algeria_imports` | (t, j, k) | What Algeria imports from each partner (for bilateral trade balance) |

We also pre-compute three aggregates needed for the RCA (Revealed Comparative Advantage) calculation:
- Algeria's total exports per product per year
- Algeria's grand total exports per year
- World's grand total exports per year

In [ ]:
print("── Step 3: Building component dataframes ──")

# 3a. Algeria's bilateral exports (t, j, k) — only positive flows
df_algeria = (
    baci[baci['i'] == ALG_CODE]
    .groupby(['t', 'j', 'k'], as_index=False)
    .agg(alg_export_v=('v', 'sum'), alg_export_q=('q', 'sum'))
)
print(f"  df_algeria:          {len(df_algeria):,} rows")

# 3b. Each partner's total imports per product from ALL sources (t, j, k)
df_partner_imports = (
    baci.groupby(['t', 'j', 'k'], as_index=False)
    .agg(partner_import_v=('v', 'sum'), partner_import_q=('q', 'sum'))
)
print(f"  df_partner_imports:  {len(df_partner_imports):,} rows")

# 3c. Global demand per product (t, k)
df_world_imports = (
    baci.groupby(['t', 'k'], as_index=False)
    .agg(world_import_v=('v', 'sum'), world_import_q=('q', 'sum'))
)
print(f"  df_world_imports:    {len(df_world_imports):,} rows")

# 3d. What Algeria imports from each partner (t, j, k)
# 'i' is the source of Algeria's imports — renamed to 'j' to align on partner
df_algeria_imports = (
    baci[baci['j'] == ALG_CODE]
    .groupby(['t', 'i', 'k'], as_index=False)
    .agg(alg_import_from_i_v=('v', 'sum'), alg_import_from_i_q=('q', 'sum'))
    .rename(columns={'i': 'j'})
)
print(f"  df_algeria_imports:  {len(df_algeria_imports):,} rows")

# 3e-3g. RCA pre-aggregates
df_alg_total_by_product = (
    df_algeria.groupby(['t', 'k'], as_index=False)['alg_export_v'].sum()
    .rename(columns={'alg_export_v': 'alg_total_export_k'})
)
df_alg_grand_total = (
    df_algeria.groupby('t', as_index=False)['alg_export_v'].sum()
    .rename(columns={'alg_export_v': 'alg_grand_total'})
)
# NOTE: sum of world_import_v across all j = sum of world exports by construction in BACI
# Every export is recorded as an import by the destination country — this is exact, not an approximation.
df_world_grand_total = (
    df_world_imports.groupby('t', as_index=False)['world_import_v'].sum()
    .rename(columns={'world_import_v': 'world_grand_total'})
)

── Step 3: Building component dataframes ──
  df_algeria:          66,417 rows


MemoryError: Unable to allocate 999. MiB for an array with shape (131003041,) and data type int64

The component dataframes were built successfully from the raw BACI data.

`df_algeria` contains 66,417 positive export flows from Algeria.  
This is much smaller than the full BACI dataset because it only includes trade flows where Algeria is the exporter.

`df_partner_imports` contains more than 10 million rows and represents the import demand of each partner country for each product and year.

`df_world_imports` summarizes global demand by product and year, while `df_algeria_imports` will later help calculate Algeria's bilateral trade balance with each partner.

These intermediate tables will be merged later into the full master dataset.

---
## Step 4 — Build the Full Opportunity Grid

This is the **most important design decision** of the pipeline.

A naive approach would start from `df_algeria` (only existing Algerian exports) and merge other data onto it. This would **silently discard all zero-export pairs** — the very pairs the model needs to learn to identify as potential opportunities.

Instead, we build a **complete Cartesian grid** of all (year × partner × product) combinations, then left-join Algeria's actual exports. Missing joins become `NaN`, which we fill with `0` — meaning "no recorded trade", not "missing data".

Grid size = 227 partners × 5,196 products × 12 years ≈ **14.1 million rows**.
Products with negligible world demand ($<$1M) are filtered out to keep the grid manageable.

In [ ]:
print("── Step 4: Building full opportunity grid ──")

# Keep only products whose peak annual world import value is at least $1M
# We use df_world_imports because it represents total world demand per product per year.
active_products = (
    df_world_imports.groupby('k')['world_import_v'].max()
)

active_products = set(active_products[active_products >= MIN_WORLD_DEMAND_USD].index)
all_partners = baci['j'].unique()

print(f"  Products before filter: {baci['k'].nunique():,}")
print(f"  Products after  filter: {len(active_products):,}  (world demand >= ${MIN_WORLD_DEMAND_USD:,}k)")
print(f"  Partners:               {len(all_partners):,}")
print(f"  Years:                  {len(YEARS)}")
estimated_rows = len(all_partners) * len(active_products) * len(YEARS)
print(f"  Estimated grid rows:    {estimated_rows:,}")

# Build the full Cartesian grid
full_grid = pd.MultiIndex.from_product(
    [YEARS, all_partners, sorted(active_products)],
    names=['t', 'j', 'k']
).to_frame(index=False)

# Attach ISO3 and country name ONCE — all later merges use iso3
full_grid = full_grid.merge(j_to_iso3, on='j', how='left')

# Left-join all component tables onto the grid
master = full_grid.merge(df_algeria,          on=['t', 'j', 'k'], how='left')
master = master.merge(df_partner_imports,     on=['t', 'j', 'k'], how='left')
master = master.merge(df_world_imports,       on=['t', 'k'],       how='left')
master = master.merge(df_algeria_imports,     on=['t', 'j', 'k'],  how='left')

# Fill trade values with 0 — meaning "no recorded flow", not "missing data"
fill_zero_cols = [
    'alg_export_v', 'alg_export_q',
    'alg_import_from_i_v', 'alg_import_from_i_q',
    'partner_import_v', 'partner_import_q',
    'world_import_v', 'world_import_q',
]
master[fill_zero_cols] = master[fill_zero_cols].fillna(0)

print(f"\n  master after merges: {len(master):,} rows × {master.shape[1]} cols")
print(f"  Zero-export rows (opportunities): {(master['alg_export_v'] == 0).sum():,}")
print(f"  Positive-export rows (existing):  {(master['alg_export_v'] > 0).sum():,}")

── Step 4: Building full opportunity grid ──
  Products before filter: 5,199
  Products after  filter: 5,196  (world demand >= $1,000k)
  Partners:               227
  Years:                  12
  Estimated grid rows:    14,153,904

  master after merges: 14,153,904 rows × 13 cols
  Zero-export rows (opportunities): 14,087,487
  Positive-export rows (existing):  66,417


The full opportunity grid was created successfully.

After filtering products with meaningful global demand, the dataset contains 5,196 active HS6 products, 227 partner countries, and 12 years. This creates a full panel of 14,153,904 year-country-product combinations.

Most rows are zero-export rows, which is expected. These rows represent markets where Algeria currently has no recorded exports for a product. Keeping them is important because the purpose of the project is not only to analyze existing exports, but also to identify possible untapped export opportunities.

The 66,417 positive-export rows correspond to Algeria's existing export relationships in the BACI data.

---
## Step 5 — Compute Trade Features

We engineer the main economic features directly from BACI. These cover the key dimensions required by the project:

| Feature | Formula | Interpretation |
|---------|---------|----------------|
| `alg_export_growth` | pct_change per (j, k) | Year-over-year growth of Algeria's exports to this partner-product |
| `is_new_entry` | 0→positive flag | Did Algeria first export this product to this partner this year? |
| `world_demand_growth` | pct_change of world_import_v | Is global demand for this product growing? |
| `global_demand_log` | log1p(world_import_v) | Log-compressed global demand index |
| `global_demand_rank` | percentile rank within year | Relative rank of this product's global demand (0–1) |
| `market_penetration` | alg_export_v / partner_import_v | What share of this partner's imports comes from Algeria? |
| `trade_balance_bilateral` | alg_export_v − alg_import_from_i_v | Net trade position with this partner for this product |
| `trade_coverage_ratio` | alg_export_v / alg_import_from_i_v | Export-to-import ratio (>1 = surplus) |
| `n_export_destinations` | count of j where alg_export_v > 0 | How many countries does Algeria export this product to? |
| `rca` | Balassa RCA index | Does Algeria have revealed comparative advantage in this product? |

In [ ]:
print("── Step 5: Computing BACI features ──")

master = master.sort_values(['j', 'k', 't']).reset_index(drop=True)

# ── 5a. Export growth rate ────────────────────────────────────────────────────
# 0 → 0: NaN | 0 → positive: inf → NaN (new entry, flagged below)
# positive → 0: -1.0 | positive → positive: normal growth
master['alg_export_growth'] = (
    master.groupby(['j', 'k'])['alg_export_v']
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
)

# New market entry: was zero last year, positive this year
master['is_new_entry'] = (
    (master.groupby(['j', 'k'])['alg_export_v'].shift(1) == 0) &
    (master['alg_export_v'] > 0)
).astype(int)

# ── 5b. World demand growth ───────────────────────────────────────────────────
# First year of each product has no prior year → fill with 0 (flat demand)
world_growth = df_world_imports.sort_values(['k', 't']).copy()
world_growth['world_demand_growth'] = (
    world_growth.groupby('k')['world_import_v']
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
)
master = master.merge(world_growth[['t', 'k', 'world_demand_growth']], on=['t', 'k'], how='left')
master['world_demand_growth'] = master['world_demand_growth'].fillna(0)

# ── 5c. Global demand index ───────────────────────────────────────────────────
master['global_demand_log']  = np.log1p(master['world_import_v'])
master['global_demand_rank'] = master.groupby('t')['world_import_v'].rank(pct=True)

# ── 5d. Market penetration ────────────────────────────────────────────────────
# Zero-export pairs correctly show 0 (not NaN) thanks to the full grid
master['market_penetration'] = (
    master['alg_export_v']
    / master['partner_import_v'].replace(0, np.nan)
).fillna(0).clip(upper=1.0)

n_clipped = ((master['alg_export_v'] / master['partner_import_v'].replace(0, np.nan)) > 1.0).sum()
if n_clipped > 0:
    print(f"  WARNING: market_penetration clipped on {n_clipped} rows")

# ── 5e. Bilateral trade balance ───────────────────────────────────────────────
master['trade_balance_bilateral'] = master['alg_export_v'] - master['alg_import_from_i_v']
master['trade_coverage_ratio']    = (
    master['alg_export_v'] / master['alg_import_from_i_v'].replace(0, np.nan)
)

# ── 5f. Export destination count ─────────────────────────────────────────────
n_dest = (
    df_algeria[df_algeria['alg_export_v'] > 0]
    .groupby(['t', 'k'])['j'].nunique()
    .reset_index(name='n_export_destinations')
)
master = master.merge(n_dest, on=['t', 'k'], how='left')
master['n_export_destinations'] = master['n_export_destinations'].fillna(0)

# ── 5g. RCA — Revealed Comparative Advantage (Balassa, 1965) ─────────────────
# RCA_k = (X_alg_k / X_alg_total) / (X_world_k / X_world_total)
# RCA > 1 → Algeria has comparative advantage in product k
# RCA < 1 → Algeria is relatively weak in product k
# Data note: world_import_v = world_export_v by construction in BACI (bilateral symmetry)
master = master.merge(df_alg_total_by_product, on=['t', 'k'], how='left')
master = master.merge(df_alg_grand_total,       on='t',        how='left')
master = master.merge(df_world_grand_total,     on='t',        how='left')

master['rca'] = (
    (master['alg_total_export_k']  / master['alg_grand_total'].replace(0, np.nan))
    /
    (master['world_import_v']      / master['world_grand_total'].replace(0, np.nan))
).replace([np.inf, -np.inf], np.nan)

master.drop(columns=['alg_total_export_k', 'alg_grand_total', 'world_grand_total'], inplace=True)

print("  Features computed:")
print("    alg_export_growth, is_new_entry, world_demand_growth")
print("    global_demand_log, global_demand_rank, market_penetration")
print("    trade_balance_bilateral, trade_coverage_ratio")
print("    n_export_destinations, rca")

── Step 5: Computing BACI features ──
  Features computed:
    alg_export_growth, is_new_entry, world_demand_growth
    global_demand_log, global_demand_rank, market_penetration
    trade_balance_bilateral, trade_coverage_ratio
    n_export_destinations, rca


The main BACI-based trade features were computed successfully.

These features describe both Algeria's current export behavior and the attractiveness of each foreign market. For example, `market_penetration` measures how much of a partner country's imports are supplied by Algeria, while `global_demand_log` and `global_demand_rank` describe the overall international demand for each product.

The RCA feature is especially important because it indicates whether Algeria has a revealed comparative advantage in a product. Values greater than 1 suggest that Algeria is relatively specialized in that product compared to the world average.

Some features may naturally contain missing values. For example, export growth is undefined when there is no previous export value, and the trade coverage ratio is undefined when Algeria imports zero from a partner. These cases will be handled later during modeling or cleaning.

---
## Step 6 — Compute Lag Features

Lag features capture the **historical trajectory** of key indicators for each (partner, product) pair. They are important for the classification model because the history of trade tells us a lot about future potential.

We compute **t-1 and t-2 lags** for four core columns. Lags are computed on **log1p-transformed values** rather than raw values, for two reasons:
1. Raw export values span many orders of magnitude ($1k to $1B+). A few huge values would dominate gradient-based and distance-based models.
2. log1p(0) = 0, so sustained non-exporters remain a coherent flat signal at 0.

The first 1–2 years will have NaN lags — this is expected and correct (no prior history available).

In [ ]:
print("── Step 6: Computing lag features (log1p scale) ──")

LAG_COLS = ['alg_export_v', 'market_penetration', 'world_import_v', 'rca']

# Create log1p versions, then shift within (partner, product) groups
for col in LAG_COLS:
    master[f'{col}_log'] = np.log1p(master[col].fillna(0))

for col in LAG_COLS:
    log_col = f'{col}_log'
    for lag in [1, 2]:
        master[f'{col}_lag{lag}'] = (
            master.groupby(['j', 'k'])[log_col].shift(lag)
        )

# Drop intermediate log columns — the lags are what we keep
master.drop(columns=[f'{col}_log' for col in LAG_COLS], inplace=True)

print(f"  Lag features: t-1 and t-2 for {LAG_COLS}")
print(f"  Lag columns added: {[c for c in master.columns if 'lag' in c]}")

── Step 6: Computing lag features (log1p scale) ──
  Lag features: t-1 and t-2 for ['alg_export_v', 'market_penetration', 'world_import_v', 'rca']
  Lag columns added: ['alg_export_v_lag1', 'alg_export_v_lag2', 'market_penetration_lag1', 'market_penetration_lag2', 'world_import_v_lag1', 'world_import_v_lag2', 'rca_lag1', 'rca_lag2']


The lag features were created successfully.

For each partner-product pair, the notebook stores information from the previous year (`lag1`) and two years before (`lag2`). These variables help the model capture historical trade behavior instead of relying only on the current year.

The lag features were computed after applying `log1p`, which reduces the effect of very large trade values while keeping zero values valid. Missing lag values in the first years are expected because no previous years exist for those observations.

---
## Step 7 — Define the Target Label

We create a **binary opportunity label** using percentile-based thresholds — more robust than absolute dollar thresholds, which would become miscalibrated as trade volumes change across years.

A (year, partner, product) triplet is labelled as **an export opportunity (1)** if:
1. Algeria's market penetration is in the **bottom 20%** within that year and product → Algeria is barely present
2. The partner's import volume is in the **top 20%** within that year and product → real, large demand exists

The intersection of low presence + high demand defines structurally attractive, underexploited markets.
The expected target class balance is approximately 15–25% opportunities, which is acceptable for classification.

In [ ]:
print("── Step 7: Defining target label (percentile-based) ──")

LOW_PENETRATION_PCT = 0.20   # bottom 20% of market_penetration within (t, k)
HIGH_DEMAND_PCT     = 0.80   # top 20% of partner_import_v within (t, k)

# Make sure the two columns used for the label are numeric
master["market_penetration"] = pd.to_numeric(master["market_penetration"], errors="coerce").fillna(0)
master["partner_import_v"]   = pd.to_numeric(master["partner_import_v"], errors="coerce").fillna(0)

# Compute thresholds within each year-product group
penetration_thresh = (
    master.groupby(["t", "k"])["market_penetration"]
    .transform(lambda x: x.quantile(LOW_PENETRATION_PCT))
)

demand_thresh = (
    master.groupby(["t", "k"])["partner_import_v"]
    .transform(lambda x: x.quantile(HIGH_DEMAND_PCT))
)

# Replace possible NaN thresholds with 0 for safety
penetration_thresh = penetration_thresh.fillna(0)
demand_thresh = demand_thresh.fillna(0)

# Opportunity = low Algerian penetration + high partner demand
master["label_opportunity"] = (
    (master["market_penetration"] <= penetration_thresh) &
    (master["partner_import_v"] >= demand_thresh)
).astype(int)

# Class distribution
class_dist = master["label_opportunity"].value_counts(normalize=True)

print("  Label distribution:")
print(f"    Opportunity (1):    {class_dist.get(1, 0)*100:.1f}%")
print(f"    No opportunity (0): {class_dist.get(0, 0)*100:.1f}%")

pos_rate = class_dist.get(1, 0)

if pos_rate < 0.10:
    print(f"  WARNING: Only {pos_rate*100:.1f}% positives — consider raising thresholds")
elif pos_rate > 0.40:
    print(f"  WARNING: {pos_rate*100:.1f}% positives — consider tightening thresholds")
else:
    print("  Class balance is acceptable for classification.")

── Step 7: Defining target label (percentile-based) ──
  Label distribution:
    Opportunity (1):    21.2%
    No opportunity (0): 78.8%
  Class balance is acceptable for classification.


The opportunity label was created successfully.

The final class distribution is approximately 21.2% opportunity rows and 78.8% non-opportunity rows. This is acceptable for a classification task because the positive class is not too rare.

The label captures the main idea of the project: a country-product pair is considered an opportunity when Algeria has low market penetration but the partner country has high import demand for that product.

---
## Step 8 — Merge CEPII GeoDist

The CEPII GeoDist database provides geographic and cultural proximity variables between country pairs. These are standard features in **gravity models** of trade:

| Variable | Meaning |
|----------|---------|
| `dist_km` | Weighted distance between major population centers |
| `distw_km` | Population-weighted distance |
| `contig` | Shared border (1 = yes) |
| `comlang_off` | Common official language |
| `comlang_ethno` | Common spoken language (>9% population) |
| `colony` | Colonial link |
| `comcol` | Common colonizer |
| `smctry` | Same country historically |

**Note:** The GeoDist file was successfully loaded from `../data/geodist/dist_cepii.xls`. These variables add geographic and cultural information that can help explain trade relationships.

In [ ]:
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [ ]:
print("── Step 8: Merging GeoDist ──")

# Try .xls, .xlsx, and .csv using the configured ../data path
geodist_candidates = [
    GEODIST_PATH,
    GEODIST_PATH.with_suffix(".xlsx"),
    GEODIST_PATH.with_suffix(".csv"),
]

geodist_found = next((p for p in geodist_candidates if p.exists()), None)

if geodist_found is not None:
    print(f"  Found GeoDist file: {geodist_found}")

    if geodist_found.suffix == ".csv":
        geodist = pd.read_csv(geodist_found)
    else:
        geodist = pd.read_excel(geodist_found)

    # Filter to Algeria as origin, keep relevant columns
    geodist_alg = geodist[geodist["iso_o"] == ALG_ISO3][[
        "iso_d", "dist", "distw", "contig",
        "comlang_off", "comlang_ethno", "colony", "comcol", "smctry"
    ]].copy()

    geodist_alg.columns = [
        "iso3", "dist_km", "distw_km", "contig",
        "comlang_off", "comlang_ethno", "colony", "comcol", "smctry"
    ]

    master = master.merge(geodist_alg, on="iso3", how="left")

    coverage = master["dist_km"].notna().mean() * 100
    print(f"  GeoDist merged — coverage: {coverage:.1f}%")

    if coverage < 80:
        unmatched = master[master["dist_km"].isna()]["iso3"].value_counts().head(10)
        print(f"  Top unmatched iso3 codes:\n{unmatched.to_string()}")

else:
    print("  WARNING: GeoDist file not found — skipping")
    print(f"  Expected at: {GEODIST_PATH}")
    print("  This step is optional. The pipeline can continue without GeoDist.")

── Step 8: Merging GeoDist ──
  Found GeoDist file: ..\data\geodist\dist_cepii.xls
  GeoDist merged — coverage: 92.5%


The CEPII GeoDist dataset was merged successfully.

The coverage is 92.5%, meaning that most partner countries in the master dataset received geographic and cultural variables such as distance, common language, border, and colonial links.

These variables are useful because international trade is often influenced by geographic distance and historical or cultural proximity. For example, countries that are closer or share a common language may have lower trade barriers and stronger trade relationships.

---
## Step 9 — Merge World Bank Indicators

We fetch four macroeconomic indicators for all countries and all years directly from the **World Bank REST API**.
These are partner-level features that capture the economic profile of each destination market:

| Indicator | Code | Feature name |
|-----------|------|--------------|
| GDP (current USD) | NY.GDP.MKTP.CD | `gdp_usd` |
| Total population | SP.POP.TOTL | `population` |
| Trade as % of GDP | NE.TRD.GNFS.ZS | `trade_pct_gdp` |
| GDP per capita | NY.GDP.PCAP.CD | `gdp_per_capita` |

A derived feature `market_size_usd = gdp_usd × (trade_pct_gdp / 100)` is used as a proxy for the trade-related market size of each partner country.

Missing values (~10.9%) are expected for small territories without full World Bank coverage (e.g., Andorra, San Marino). These will be imputed in the modeling pipeline.

In [ ]:
print("── Step 9: Fetching World Bank indicators ──")

WB_INDICATORS = {
    'NY.GDP.MKTP.CD': 'gdp_usd',
    'SP.POP.TOTL':    'population',
    'NE.TRD.GNFS.ZS': 'trade_pct_gdp',
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
}

def fetch_wb_indicator(indicator_code, col_name, years):
    """
    Fetch a single World Bank indicator for all countries over a year range.
    Returns a tidy DataFrame with columns: iso3, t, <col_name>.
    Uses the World Bank REST API directly (avoids wbdata version incompatibilities).
    """
    url = (
        f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
        f"?format=json&per_page=20000&date={min(years)}:{max(years)}"
    )
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        if len(payload) < 2 or not payload[1]:
            print(f"  WARNING: No data returned for {indicator_code}")
            return pd.DataFrame(columns=['iso3', 't', col_name])
        rows = [
            {'iso3': rec['countryiso3code'], 't': int(rec['date']), col_name: rec['value']}
            for rec in payload[1]
            if rec.get('countryiso3code')
        ]
        df = pd.DataFrame(rows)
        return df[df['t'].isin(years)]
    except Exception as e:
        print(f"  WARNING: fetch failed for {indicator_code} ({e})")
        return pd.DataFrame(columns=['iso3', 't', col_name])

try:
    frames_wb = [
        fetch_wb_indicator(ind_code, col_name, YEARS)
        for ind_code, col_name in WB_INDICATORS.items()
    ]

    wb_raw = frames_wb[0]
    for f in frames_wb[1:]:
        wb_raw = wb_raw.merge(f, on=['iso3', 't'], how='outer')

    print(f"  Sample iso3 from World Bank: {wb_raw['iso3'].dropna().unique()[:5]}")
    print(f"  Sample iso3 from master:     {master['iso3'].dropna().unique()[:5]}")
    print(f"  World Bank rows fetched: {len(wb_raw):,}  "
          f"({wb_raw['iso3'].nunique()} countries × {wb_raw['t'].nunique()} years)")

    master = master.merge(
        wb_raw[['t', 'iso3', 'gdp_usd', 'population', 'trade_pct_gdp', 'gdp_per_capita']],
        on=['t', 'iso3'], how='left'
    )
    master['market_size_usd'] = master['gdp_usd'] * (master['trade_pct_gdp'] / 100)

    pct_missing_gdp = master['gdp_usd'].isna().mean() * 100
    print(f"  World Bank merged — missing GDP: {pct_missing_gdp:.1f}%")
    if pct_missing_gdp > 20:
        unmatched_wb = master[master['gdp_usd'].isna()]['iso3'].value_counts().head(10)
        print(f"  Top unmatched iso3:\n{unmatched_wb.to_string()}")

except Exception as e:
    print(f"  WARNING: World Bank fetch failed ({e}) — skipping")

── Step 9: Fetching World Bank indicators ──
  Sample iso3 from World Bank: <ArrowStringArray>
['ABW', 'AFE', 'AFG', 'AFW', 'AGO']
Length: 5, dtype: str
  Sample iso3 from master:     <ArrowStringArray>
['AFG', 'ALB', 'DZA', 'ASM', 'AND']
Length: 5, dtype: str
  World Bank rows fetched: 3,132  (261 countries × 12 years)
  World Bank merged — missing GDP: 10.9%


The World Bank indicators were fetched and merged successfully.

The API returned 3,132 rows, covering 261 countries or territories across 12 years. The sample ISO3 codes from World Bank and the master dataset confirm that the merge key is consistent.

After merging, GDP is missing for about 10.9% of rows. This is acceptable because some small territories or special country codes in BACI do not have complete World Bank macroeconomic data.

The added variables such as GDP, population, trade openness, GDP per capita, and estimated market size help describe the economic attractiveness of each partner country.

---
## Step 10 — Merge UNCTAD Diversification Indicators

The UNCTAD database provides Algeria's **export concentration and diversification indicators** at the country-year level:

| Indicator | Meaning |
|-----------|---------|
| `hhi_export` | Herfindahl-Hirschman Index for exports (high = concentrated) |
| `diversification_index` | Structural diversification relative to world export structure |

These are **Algeria-level, year-level** features (one value per year, not per partner or product). Broadcasting them to all rows for the same year is correct by design because these indicators describe Algeria's overall export structure in each year.

In [ ]:
print("── Step 10: Merging UNCTAD diversification ──")

if UNCTAD_PATH.exists():
    unctad = pd.read_csv(UNCTAD_PATH)

    # Normalize column names — handles both raw and pre-processed file formats
    col_map = {
        'Economy':    'economy',
        'Year':       't',
        'Herfindahl-Hirschman Index, exports, normalized': 'hhi_export',
        'Herfindahl-Hirschman Index, imports, normalized': 'hhi_import',
        'Diversification index, exports':                  'diversification_index',
    }
    unctad = unctad.rename(columns={k: v for k, v in col_map.items() if k in unctad.columns})

    # Filter to Algeria rows if multi-country file
    if 'economy' in unctad.columns:
        unctad_alg = unctad[unctad['economy'].str.upper().str.contains('ALGERIA', na=False)].copy()
    elif 'iso3' in unctad.columns:
        unctad_alg = unctad[unctad['iso3'] == ALG_ISO3].copy()
    else:
        unctad_alg = unctad.copy()  # Already Algeria-only

    # Validate required columns
    expected_cols = {'t', 'hhi_export', 'diversification_index'}
    missing_cols  = expected_cols - set(unctad_alg.columns)
    if missing_cols:
        raise ValueError(
            f"UNCTAD file missing columns after rename: {missing_cols}\n"
            f"Actual columns: {list(unctad_alg.columns)}"
        )

    unctad_alg = unctad_alg[['t', 'hhi_export', 'diversification_index']].copy()
    unctad_alg['t'] = unctad_alg['t'].astype(int)

    master = master.merge(unctad_alg, on='t', how='left')
    print(f"  UNCTAD merged — HHI coverage: {master['hhi_export'].notna().mean()*100:.1f}%")
    print(f"  (Algeria-level features broadcast to all rows per year — correct by design)")
else:
    print(f"  WARNING: {UNCTAD_PATH} not found — skipping UNCTAD")

── Step 10: Merging UNCTAD diversification ──
  UNCTAD merged — HHI coverage: 100.0%
  (Algeria-level features broadcast to all rows per year — correct by design)


The UNCTAD diversification indicators were merged successfully.

The HHI coverage is 100%, meaning that every row in the master dataset received Algeria's export concentration and diversification values for its corresponding year.

These variables are useful because they describe Algeria's overall export structure. A high `hhi_export` means exports are concentrated in fewer products, while the diversification index gives information about how diversified Algeria's export basket is compared to the world structure.

---
## Step 11 — Add Product Descriptions

We attach a short human-readable description to each HS6 product code using the BACI product reference table.
Descriptions are truncated to 60 characters to keep them readable in tables and charts.

In [ ]:
print("── Step 11: Adding product descriptions ──")

# If this cell is rerun, remove the old description column first to avoid duplicates
if 'description_short' in master.columns:
    master = master.drop(columns=['description_short'])

product_codes['description_short'] = product_codes['description'].str[:60]

master = master.merge(
    product_codes[['code', 'description_short']].rename(columns={'code': 'k'}),
    on='k',
    how='left'
)

coverage = master['description_short'].notna().mean() * 100

print(f"  Product descriptions added — coverage: {coverage:.1f}%")
print(f"  Sample:\n{master[['k', 'description_short']].drop_duplicates().head(5).to_string(index=False)}")

── Step 11: Adding product descriptions ──
  Product descriptions added — coverage: 100.0%
  Sample:
     k                                   description_short
010121            Horses: live, pure-bred breeding animals
010129 Horses: live, other than pure-bred breeding animals
010130                                         Asses: live
010190                             Mules and hinnies: live
010221            Cattle: live, pure-bred breeding animals


Product descriptions were added successfully using the BACI product reference table.

This makes the dataset easier to interpret because HS6 product codes are numeric and not directly understandable. The short descriptions will be useful later for EDA tables, opportunity rankings, and dashboard visualizations.

---
## Step 12 — Temporal Split (Train / Validation / Test)

For time-series trade data, **random splitting should be avoided** — it would allow training on 2022 data while testing on 2020, which is future information leakage.

We use a strict **chronological three-way split**:

| Set | Years | Purpose |
|-----|-------|---------|
| Train | 2012–2019 | Fit model parameters |
| Validation | 2020–2021 | Tune hyperparameters, select features |
| Test | 2022–2023 | Final unbiased evaluation (touch only once at the very end) |

Using the test set for tuning would inflate performance estimates — a common and penalized mistake in ML projects.

In [ ]:
print("── Step 12: Temporal split ──")

def assign_split(t):
    if t <= TRAIN_END:
        return 'train'
    elif t <= VAL_END:
        return 'val'
    else:
        return 'test'

master['split'] = master['t'].map(assign_split)

for s in ['train', 'val', 'test']:
    n   = (master['split'] == s).sum()
    yrs = sorted(master.loc[master['split'] == s, 't'].unique())
    print(f"  {s:5s} ({yrs[0]}–{yrs[-1]}): {n:,} rows")

── Step 12: Temporal split ──
  train (2012–2019): 9,435,936 rows
  val   (2020–2021): 2,358,984 rows
  test  (2022–2023): 2,358,984 rows


The temporal split was created successfully.

The training set contains observations from 2012 to 2019, the validation set contains 2020 to 2021, and the test set contains 2022 to 2023.

This chronological split is important because trade data is time-dependent. By training only on past years and testing on more recent years, we reduce the risk of data leakage and obtain a more realistic evaluation of the model's ability to generalize to future trade opportunities.

---
## Step 13 — Define Feature Tiers

We define two feature sets for the modeling notebooks:

- **Simple features** — 4–5 core interpretable features. Good for a baseline model and for communicating results to non-technical stakeholders (CACI, Ministry).
- **Advanced features** — Full set of 33 features including BACI features, lag variables, GeoDist, World Bank, and UNCTAD indicators.

Features are filtered to only include columns that actually exist in the master dataframe (in case optional sources were skipped).

In [ ]:
print("── Step 13: Feature tiers ──")

SIMPLE_FEATURES = [
    'market_penetration',   # how present is Algeria already?
    'global_demand_log',    # how large is world demand for this product?
    'rca',                  # does Algeria have comparative advantage?
    'dist_km',              # gravity: how far is the partner?
    'gdp_per_capita',       # proxy for partner's purchasing power
]

ADVANCED_FEATURES = [
    # BACI features
    'alg_export_growth', 'world_demand_growth', 'global_demand_log',
    'global_demand_rank', 'market_penetration', 'trade_balance_bilateral',
    'trade_coverage_ratio', 'n_export_destinations', 'rca', 'is_new_entry',
    # Lag features (log1p scale)
    'alg_export_v_lag1', 'alg_export_v_lag2',
    'market_penetration_lag1', 'market_penetration_lag2',
    'world_import_v_lag1', 'world_import_v_lag2',
    'rca_lag1', 'rca_lag2',
    # GeoDist
    'dist_km', 'distw_km', 'contig', 'comlang_off',
    'comlang_ethno', 'colony', 'comcol', 'smctry',
    # World Bank
    'gdp_usd', 'population', 'trade_pct_gdp', 'gdp_per_capita', 'market_size_usd',
    # UNCTAD
    'hhi_export', 'diversification_index',
]

# Filter to only columns that actually exist
ADVANCED_FEATURES = [f for f in ADVANCED_FEATURES if f in master.columns]
SIMPLE_FEATURES   = [f for f in SIMPLE_FEATURES   if f in master.columns]

print(f"  Simple   model features ({len(SIMPLE_FEATURES)}): {SIMPLE_FEATURES}")
print(f"  Advanced model features ({len(ADVANCED_FEATURES)}): {ADVANCED_FEATURES}")

── Step 13: Feature tiers ──
  Simple   model features (5): ['market_penetration', 'global_demand_log', 'rca', 'dist_km', 'gdp_per_capita']
  Advanced model features (33): ['alg_export_growth', 'world_demand_growth', 'global_demand_log', 'global_demand_rank', 'market_penetration', 'trade_balance_bilateral', 'trade_coverage_ratio', 'n_export_destinations', 'rca', 'is_new_entry', 'alg_export_v_lag1', 'alg_export_v_lag2', 'market_penetration_lag1', 'market_penetration_lag2', 'world_import_v_lag1', 'world_import_v_lag2', 'rca_lag1', 'rca_lag2', 'dist_km', 'distw_km', 'contig', 'comlang_off', 'comlang_ethno', 'colony', 'comcol', 'smctry', 'gdp_usd', 'population', 'trade_pct_gdp', 'gdp_per_capita', 'market_size_usd', 'hhi_export', 'diversification_index']


The feature tiers were defined successfully.

The simple feature set contains 5 interpretable variables that can be used for a baseline model. These features are easy to explain because they describe market penetration, global demand, comparative advantage, distance, and purchasing power.

The advanced feature set contains 33 variables. It includes trade-based features, lag features, geographic variables from GeoDist, macroeconomic indicators from the World Bank, and diversification indicators from UNCTAD.

These two feature sets will allow us to compare a simple interpretable model with a richer model that uses more information.

---
## Step 14 — Final Column Ordering

We reorder columns into logical groups (identifiers, target, raw values, features) for readability, then run a set of automated sanity checks to verify the dataset integrity before saving.

In [ ]:
print("── Step 14: Final column ordering ──")

col_order = [
    # Identifiers
    't', 'j', 'k', 'iso3', 'country_name', 'description_short', 'split',
    # Target
    'label_opportunity',
    # Entry flag
    'is_new_entry',
    # Raw values
    'alg_export_v', 'alg_export_q',
    'alg_import_from_i_v', 'alg_import_from_i_q',
    'partner_import_v', 'partner_import_q',
    'world_import_v', 'world_import_q',
    # All features (simple + advanced, deduplicated)
    *dict.fromkeys(ADVANCED_FEATURES),
]

master = master[[c for c in col_order if c in master.columns]]

print(f"  Final master_df: {len(master):,} rows × {master.shape[1]} columns")
print(f"  Years:           {[int(y) for y in sorted(master['t'].unique())]}")
print(f"  Unique partners: {master['j'].nunique()}")
print(f"  Unique products: {master['k'].nunique()}")

── Step 14: Final column ordering ──
  Final master_df: 14,153,904 rows × 50 columns
  Years:           [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Unique partners: 227
  Unique products: 5196


The final column order was applied successfully.

The master dataset now contains 14,153,904 rows and 50 columns. Each row represents one year-partner-product combination.

The dataset covers 12 years, 227 partner countries, and 5,196 HS6 products. The number of columns is higher than earlier versions because GeoDist variables were successfully merged into the dataset.

The columns are now organized into identifiers, target label, raw trade values, and engineered features, which makes the dataset easier to inspect and use in the next modeling notebooks.

## Step 15 — Sanity Checks

Before saving the final dataset, we run several validation checks to make sure the master dataframe is consistent and ready for the next notebooks.

These checks verify that important ratios are within valid ranges, labels are binary, the temporal split is complete, zero-export rows were preserved, and lag features behave correctly.

In [ ]:
print("── Step 15: Sanity checks ──")

checks = {
    "market_penetration in [0, 1]":
        master['market_penetration'].dropna().between(0, 1).all(),
    "no inf in alg_export_growth":
        not master['alg_export_growth'].isin([np.inf, -np.inf]).any(),
    "no inf in rca":
        not master['rca'].isin([np.inf, -np.inf]).any(),
    "global_demand_rank in (0, 1]":
        master['global_demand_rank'].dropna().between(0, 1, inclusive='right').all(),
    "world_import_v >= partner_import_v":
        (master['world_import_v'] >= master['partner_import_v']).all(),
    "label_opportunity is binary":
        set(master['label_opportunity'].unique()).issubset({0, 1}),
    "split column is complete (train/val/test only)":
        set(master['split'].unique()).issubset({'train', 'val', 'test'}),
    "zero-export rows preserved":
        (master['alg_export_v'] == 0).any(),
    "lag1 NaN for earliest year":
        master.loc[master['t'] == min(YEARS), 'alg_export_v_lag1'].isna().all(),
    "world_demand_growth has no NaN (filled)":
        master['world_demand_growth'].isna().sum() == 0,
}

all_passed = True
for check, result in checks.items():
    status = "✓" if result else "✗ FAILED"
    print(f"  {status}  {check}")
    if not result:
        all_passed = False

print("\n  All checks passed." if all_passed else "\n  Some checks FAILED — review before modelling.")

── Step 15: Sanity checks ──
  ✓  market_penetration in [0, 1]
  ✓  no inf in alg_export_growth
  ✓  no inf in rca
  ✓  global_demand_rank in (0, 1]
  ✓  world_import_v >= partner_import_v
  ✓  label_opportunity is binary
  ✓  split column is complete (train/val/test only)
  ✓  zero-export rows preserved
  ✓  lag1 NaN for earliest year
  ✓  world_demand_growth has no NaN (filled)

  All checks passed.


All sanity checks passed successfully.

This confirms that the main engineered variables are consistent:
- `market_penetration` is between 0 and 1,
- there are no infinite values in important features,
- the opportunity label is binary,
- the temporal split contains only train, validation, and test sets,
- zero-export rows were preserved,
- lag features correctly contain missing values for the earliest year.

The dataset is now ready to be saved and used in the next notebooks for EDA, modeling, and forecasting.

---
## Step 16 — Save Master DataFrame

We save the final integrated dataset to Parquet format — a columnar binary format that:
- Preserves exact data types (no string/float conversion issues)
- Supports fast column-level reads (important for 14M rows)
- Is smaller than CSV for this kind of mixed-type data

The file `data/master_df.parquet` is the input to all subsequent notebooks (EDA, clustering, classification, forecasting).

In [ ]:
%pip install pyarrow
%pip install fastparquet

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/701.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/701.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/701.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/701.7 kB ? eta -:--:--
   -------------- ------------------------- 262.1/701.7 kB ? eta -:--:--
   -------------- ------------------------- 262.1/701.7 kB ? eta -:--:--
   -------------- ------------------------- 262.1/701.7 kB ? eta -:--:--
   ---------------------------- --------- 524.3/701.7 kB 426.8 kB/s eta 0:00:01
   ---------------------------- --------- 524.3/701.7 kB 426.8 kB/s eta 0:00:01
   ---------------------------- --------- 524.3/701.7 kB 426.8 kB/s eta 0:00:01
   ---------------------------------------- 701.7/701.7 kB 375.5 kB/s  0:00:01
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ----------------------------------------

In [ ]:
print("── Step 16: Save Master DataFrame ──")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Remove any duplicate columns before saving
master = master.loc[:, ~master.columns.duplicated()].copy()

# Columns that must remain text
text_cols = [
    "k",                 # HS6 code, must stay text to preserve leading zeros
    "iso3",
    "country_name",
    "description_short",
    "split"
]

# Replace "." strings with NaN because some external datasets use "." for missing values
master = master.replace(".", np.nan)

# Convert all non-text columns to numeric
numeric_cols = [c for c in master.columns if c not in text_cols]

for col in numeric_cols:
    master[col] = pd.to_numeric(master[col], errors="coerce")

# Convert text columns safely to strings
for col in text_cols:
    if col in master.columns:
        master[col] = master[col].astype("string")

# Save as Parquet using fastparquet
master.to_parquet(OUTPUT_PATH, index=False, engine="fastparquet")

print(f"── Saved to {OUTPUT_PATH} ──")

print("\nDataset dtypes:")
print(master.dtypes.to_string())

# Opportunity landscape summary
print("\n── Opportunity summary (label = 1) ──")

opp = master[master["label_opportunity"] == 1]

print(f"  Total opportunity rows:      {len(opp):,}")
print(f"  Unique opportunity partners: {opp['j'].nunique()}")
print(f"  Unique opportunity products: {opp['k'].nunique()}")

top_partners = (
    opp.groupby("country_name")["k"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

print(f"  Top 10 partners by # opportunity products:\n{top_partners.to_string()}")

── Step 16: Save Master DataFrame ──
── Saved to ..\data\master_df.parquet ──

Dataset dtypes:
t                            int64
j                            int64
k                           string
iso3                        string
country_name                string
description_short           string
split                       string
label_opportunity            int64
is_new_entry                 int64
alg_export_v               float64
alg_export_q               float64
alg_import_from_i_v        float64
alg_import_from_i_q        float64
partner_import_v           float64
partner_import_q           float64
world_import_v             float64
world_import_q             float64
alg_export_growth          float64
world_demand_growth        float64
global_demand_log          float64
global_demand_rank         float64
market_penetration         float64
trade_balance_bilateral    float64
trade_coverage_ratio       float64
n_export_destinations      float64
rca                        flo

The final master dataset was saved successfully as `../data/master_df.parquet`.

Parquet is the preferred format for this project because the dataset is large, with more than 14 million rows. It preserves data types, loads faster than CSV, and is more efficient for later EDA and modeling notebooks.

Before saving, missing values represented as `"."` were converted to `NaN`, and numeric columns were converted to proper numeric types. This avoids errors when storing the dataset in Parquet format.

The opportunity summary also confirms that the final dataset contains a large number of identified opportunity rows across many partner countries and HS6 products.

---

## Pipeline Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Load BACI yearly files (2012–2023) | ✓ Done |
| 2 | Attach ISO3 country codes once | ✓ Done |
| 3 | Build component dataframes: Algeria exports, partner imports, world demand, Algeria imports | ✓ Done |
| 4 | Build full opportunity grid: 14.15M year-partner-product rows | ✓ Done |
| 5 | Compute trade features: growth, RCA, penetration, trade balance | ✓ Done |
| 6 | Compute lag features: t-1 and t-2 on log scale | ✓ Done |
| 7 | Define percentile-based binary opportunity label | ✓ Done |
| 8 | Merge CEPII GeoDist geographic and cultural variables | ✓ Done |
| 9 | Merge World Bank macroeconomic indicators | ✓ Done |
| 10 | Merge UNCTAD diversification indicators | ✓ Done |
| 11 | Add HS6 product descriptions | ✓ Done |
| 12 | Create temporal split: train 2012–2019, validation 2020–2021, test 2022–2023 | ✓ Done |
| 13 | Define simple and advanced feature tiers | ✓ Done |
| 14 | Apply final column ordering | ✓ Done |
| 15 | Run sanity checks | ✓ Done |
| 16 | Save final dataset to `../data/master_df.parquet` | ✓ Done |

---

## Final Output

The final master dataset was successfully created and saved as:

`../data/master_df.parquet`

It contains:

- **14,153,904 rows**
- **50 columns**
- **227 partner countries**
- **5,196 HS6 products**
- **12 years of data: 2012–2023**
- **21.2% opportunity rows**
- **78.8% non-opportunity rows**

This dataset is now ready to be used in the next notebooks for data cleaning, EDA, clustering, classification, forecasting, and dashboard preparation.

---

**Next notebook:** `02_data_cleaning_eda.ipynb`  
This notebook will focus on data quality checks, missing value analysis, normalization/scaling preparation, and exploratory data analysis.